В исходном train loop были проблемы с памятью, синхронизацией CPU/GPU и измерением времени. Ниже переписан вариант, где данные переносятся на GPU аккуратно, лишние синхронизации убраны, а loss не хранит граф вычислений.

In [1]:
import statistics
import time

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [2]:
def prepare_data() -> TensorDataset:
    X = torch.randn(10000, 128)
    y = torch.randint(0, 2, (10000,))
    return TensorDataset(X, y)

In [ ]:
def train(
    batch_size: int = 256,
    lr: float = 1e-3,
    log_every: int = 20,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataloader = DataLoader(
        prepare_data(),
        batch_size=batch_size,
        shuffle=True,
        pin_memory=(device.type == "cuda"),
        num_workers=2,
        persistent_workers=True,
    )

    model = nn.Sequential(
        nn.Linear(128, 512), nn.ReLU(),
        nn.Linear(512, 128), nn.ReLU(),
        nn.Linear(128, 2),
    ).to(device).train()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    forward_times_ms = []
    backward_times_ms = []

    # Метрики считаем на GPU и переводим в Python только в конце эпохи
    total_loss = torch.zeros((), device=device)
    total_objects = 0

    for batch_idx, (data, target) in enumerate(dataloader):
        data = data.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)

        # Не создаем tensor на CPU с последующим .to("cuda")
        noise = torch.randn_like(data)
        data = data + noise

        # set_to_none=True не зануляет память, а освобождает ссылки на градиенты
        optimizer.zero_grad(set_to_none=True)

        if device.type == "cuda":
            fwd_start = torch.cuda.Event(enable_timing=True)
            fwd_end = torch.cuda.Event(enable_timing=True)
            bwd_start = torch.cuda.Event(enable_timing=True)
            bwd_end = torch.cuda.Event(enable_timing=True)

            fwd_start.record()
            output = model(data)
            loss = criterion(output, target)
            fwd_end.record()

            bwd_start.record()
            loss.backward()
            bwd_end.record()

            optimizer.step()

            # Синхронизация нужна только для честного замера времени
            torch.cuda.synchronize()
            forward_times_ms.append(fwd_start.elapsed_time(fwd_end))
            backward_times_ms.append(bwd_start.elapsed_time(bwd_end))
        else:
            t0 = time.perf_counter()
            output = model(data)
            loss = criterion(output, target)
            t1 = time.perf_counter()

            loss.backward()
            t2 = time.perf_counter()

            optimizer.step()

            forward_times_ms.append((t1 - t0) * 1000)
            backward_times_ms.append((t2 - t1) * 1000)

        bs = data.size(0)

        # detach() нужен, чтобы не хранить весь computation graph
        total_loss += loss.detach() * bs
        total_objects += bs

        # .item() вызывает синхронизацию, поэтому не делаем это на каждом batch
        if batch_idx % log_every == 0:
            print(f"Batch {batch_idx} processed")

        # torch.cuda.empty_cache() здесь не нужен: он ломает нормальную работу caching allocator

    avg_loss = (total_loss / total_objects).item()

    print(
        f"Epoch finished, avg loss: {avg_loss:.4f}, "
        f"avg forward time: {statistics.mean(forward_times_ms):.3f} ms, "
        f"avg backward time: {statistics.mean(backward_times_ms):.3f} ms"
    )

    return {
        "avg_loss": avg_loss,
        "avg_forward_ms": statistics.mean(forward_times_ms),
        "avg_backward_ms": statistics.mean(backward_times_ms),
    }

In [4]:
metrics = train()
metrics

Batch 0 processed
Batch 20 processed
Epoch finished, avg loss: 0.6977, avg forward time: 126.162 ms, avg backward time: 2.680 ms


{'avg_loss': 0.6977364420890808,
 'avg_forward_ms': 126.1620946187526,
 'avg_backward_ms': 2.679782348871231}